# Loading and Preprocessing:

In [2]:
import pandas as pd
G_train = pd.read_csv("GoEmotions/Processed/GoEmotions_train.csv")
G_test = pd.read_csv("GoEmotions/Processed/GoEmotions_test.csv")

S_train = pd.read_csv("SemEvalDataset/processed/train.csv")
S_test = pd.read_csv("SemEvalDataset/processed/test.csv")

T_train = pd.read_csv("TwitterEmotions/training.csv")
T_test = pd.read_csv("TwitterEmotions/test.csv")

In [56]:
emotions = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude",
    "grief", "joy", "love", "nervousness", "optimism", "pride",
    "realization", "relief", "remorse", "sadness", "surprise", "neutral"
]

def parse_labels(label_str):
    return [int(x) for x in str(label_str).split(",")]

def make_multilabel_columns(df):
    df = df.copy()

    # Convert "13,18" into [13, 18]
    df["label_list"] = df["label_id"].apply(parse_labels)

    # Add readable labels: [13,18] -> ["excitement", "love"]
    df["label_names"] = df["label_list"].apply(
        lambda ids: [emotions[i] for i in ids]
    )

    # Create one binary column per emotion
    for i, emotion in enumerate(emotions):
        df[emotion] = df["label_list"].apply(lambda labels: 1 if i in labels else 0)

    return df

G_train = make_multilabel_columns(G_train)
G_test = make_multilabel_columns(G_test)

# print(goemotions_train[["label_id", "label_list", "label_names"]].head())

emotion_cols = emotions
print(G_train.columns)
print(G_test.columns)

Index(['text', 'label_id', 'example_id', 'label_list', 'label_names',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
       'relief', 'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='str')
Index(['text', 'label_id', 'example_id', 'label_list', 'label_names',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
       'relief', 'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='str')


GoEmotions Statistics

In [58]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import StandardScaler

stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize_text(text):
    return re.findall(r"\b[a-z]+\b", clean_text(text))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

def pad_tokens(tokens, max_len=100, padding="post"):
    tokens = tokens[:max_len]
    pad_amount = max_len - len(tokens)

    if padding == "post":
        return tokens + ["<PAD>"] * pad_amount
    return ["<PAD>"] * pad_amount + tokens

In [59]:
def preprocess_only(df, text_col, max_len=100):
    df = df.copy()

    df["clean_text"] = df[text_col].apply(clean_text)

    df["tokens"] = df[text_col].apply(tokenize_text)

    df["tokens_no_stopwords"] = df["tokens"].apply(remove_stopwords)

    df["padded_tokens"] = df["tokens_no_stopwords"].apply(
        lambda tokens: pad_tokens(tokens, max_len=max_len, padding="post")
    )

    df["word_count"] = df["tokens"].apply(len)
    df["clean_word_count"] = df["tokens_no_stopwords"].apply(len)
    df["char_count"] = df["clean_text"].apply(len)

    scaler = StandardScaler()
    df[["word_count_scaled", "clean_word_count_scaled", "char_count_scaled"]] = scaler.fit_transform(
        df[["word_count", "clean_word_count", "char_count"]]
    )

    return df

In [60]:
G_train = G_train.rename(columns={'0': "text", '1': "label_id", '2': "example_id"})
G_test = G_test.rename(columns={'0': "text", '1': "label_id", '2': "example_id"})

In [61]:
twitter_train = preprocess_only(T_train, text_col="text", max_len=100)
twitter_test = preprocess_only(T_test, text_col="text", max_len=100)

semeval_train = preprocess_only(S_train, text_col="Tweet", max_len=100)
semeval_test = preprocess_only(S_test, text_col="Tweet", max_len=100)

goemotions_train = preprocess_only(G_train, text_col="text", max_len=100)
goemotions_test = preprocess_only(G_test, text_col="text", max_len=100)

In [62]:
print(goemotions_test.columns)
print(goemotions_train.columns)

Index(['text', 'label_id', 'example_id', 'label_list', 'label_names',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
       'relief', 'remorse', 'sadness', 'surprise', 'neutral', 'clean_text',
       'tokens', 'tokens_no_stopwords', 'padded_tokens', 'word_count',
       'clean_word_count', 'char_count', 'word_count_scaled',
       'clean_word_count_scaled', 'char_count_scaled'],
      dtype='str')
Index(['text', 'label_id', 'example_id', 'label_list', 'label_names',
       'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
       'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
       'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
       'joy', 'love', 'nervousness', 

In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

X_train_text = vectorizer.fit_transform(
    goemotions_train["clean_text"]
)

X_test_text = vectorizer.transform(
    goemotions_test["clean_text"]
)

from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix

metadata_cols = ["clean_word_count", "char_count"]

scaler = StandardScaler()

X_train_meta = scaler.fit_transform(goemotions_train[metadata_cols])
X_test_meta = scaler.transform(goemotions_test[metadata_cols])

X_train = hstack([X_train_text, csr_matrix(X_train_meta)])
X_test = hstack([X_test_text, csr_matrix(X_test_meta)])

from scipy.sparse import hstack

X_train = hstack([X_train_text, X_train_meta])
Y_train = goemotions_train[emotions]
X_test = hstack([X_test_text, X_test_meta])
Y_test = goemotions_test[emotions]

# Model Training

## Simpler Models

In [64]:
import time
import pandas as pd

from sklearn.svm import LinearSVC

from sklearn.multioutput import MultiOutputClassifier

from sklearn.metrics import classification_report, f1_score, accuracy_score

Model Training Helper Method

In [65]:
def train_and_evaluate_model(model, model_name, X_train, Y_train, X_test, Y_test):
    start_train = time.time()
    model.fit(X_train, Y_train)
    train_time = time.time() - start_train

    start_pred = time.time()
    Y_pred = model.predict(X_test)
    inference_time = time.time() - start_pred

    micro_f1 = f1_score(Y_test, Y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(Y_test, Y_pred, average="macro", zero_division=0)
    subset_accuracy = accuracy_score(Y_test, Y_pred)

    print(f"\n===== {model_name} =====")
    print(f"Training Time: {train_time:.2f} seconds")
    print(f"Inference Time: {inference_time:.2f} seconds")
    print(f"Subset Accuracy: {subset_accuracy:.4f}")
    print(f"Micro F1: {micro_f1:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")

    print("\nClassification Report:")
    print(classification_report(
        Y_test,
        Y_pred,
        target_names=emotions,
        zero_division=0
    ))

    return {
        "model_name": model_name,
        "model": model,
        "train_time": train_time,
        "inference_time": inference_time,
        "subset_accuracy": subset_accuracy,
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "Y_pred": Y_pred
    }

Naive Bayes

In [66]:
from sklearn.model_selection import GridSearchCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB

nb_model = OneVsRestClassifier(
    MultinomialNB()
)

nb_results = train_and_evaluate_model(
    nb_model,
    "Naive Bayes",
    X_train_text,
    Y_train,
    X_test_text,
    Y_test
)


===== Naive Bayes =====
Training Time: 0.25 seconds
Inference Time: 0.03 seconds
Subset Accuracy: 0.1054
Micro F1: 0.1791
Macro F1: 0.0588

Classification Report:
                precision    recall  f1-score   support

    admiration       0.90      0.10      0.19       504
     amusement       0.86      0.02      0.04       264
         anger       1.00      0.02      0.03       198
     annoyance       0.00      0.00      0.00       320
      approval       1.00      0.01      0.01       351
        caring       0.00      0.00      0.00       135
     confusion       0.00      0.00      0.00       153
     curiosity       1.00      0.02      0.03       284
        desire       0.00      0.00      0.00        83
disappointment       0.00      0.00      0.00       151
   disapproval       0.00      0.00      0.00       267
       disgust       1.00      0.02      0.03       123
 embarrassment       0.00      0.00      0.00        37
    excitement       0.00      0.00      0.00      

SVM Model

In [67]:
from sklearn.multiclass import OneVsRestClassifier

svm_model = OneVsRestClassifier(
    LinearSVC()
)

svm_results = train_and_evaluate_model(
    svm_model,
    "Linear SVM",
    X_train,
    Y_train,
    X_test,
    Y_test
)


===== Linear SVM =====
Training Time: 5.12 seconds
Inference Time: 0.03 seconds
Subset Accuracy: 0.3510
Micro F1: 0.4896
Macro F1: 0.3453

Classification Report:
                precision    recall  f1-score   support

    admiration       0.70      0.48      0.57       504
     amusement       0.82      0.59      0.69       264
         anger       0.69      0.20      0.31       198
     annoyance       0.49      0.10      0.17       320
      approval       0.52      0.14      0.22       351
        caring       0.50      0.16      0.25       135
     confusion       0.50      0.16      0.25       153
     curiosity       0.46      0.17      0.24       284
        desire       0.52      0.19      0.28        83
disappointment       0.62      0.07      0.12       151
   disapproval       0.36      0.10      0.15       267
       disgust       0.65      0.28      0.39       123
 embarrassment       0.33      0.08      0.13        37
    excitement       0.67      0.23      0.35       

Random Forest

In [68]:
from sklearn.ensemble import RandomForestClassifier

rf_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=30,
        random_state=42,
        n_jobs=-1
    )
)

rf_results = train_and_evaluate_model(
    rf_model,
    "Random Forest",
    X_train,
    Y_train,
    X_test,
    Y_test
)


===== Random Forest =====
Training Time: 19.86 seconds
Inference Time: 0.79 seconds
Subset Accuracy: 0.0302
Micro F1: 0.0683
Macro F1: 0.0338

Classification Report:
                precision    recall  f1-score   support

    admiration       1.00      0.01      0.01       504
     amusement       1.00      0.02      0.04       264
         anger       1.00      0.03      0.05       198
     annoyance       0.00      0.00      0.00       320
      approval       1.00      0.00      0.01       351
        caring       0.00      0.00      0.00       135
     confusion       0.00      0.00      0.00       153
     curiosity       1.00      0.00      0.01       284
        desire       0.00      0.00      0.00        83
disappointment       0.00      0.00      0.00       151
   disapproval       0.00      0.00      0.00       267
       disgust       1.00      0.02      0.03       123
 embarrassment       0.00      0.00      0.00        37
    excitement       0.50      0.01      0.02   

In [69]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

lr_model = OneVsRestClassifier(
    LogisticRegression(
        max_iter=1000,
        n_jobs=-1
    )
)

lr_results = train_and_evaluate_model(
    lr_model,
    "Logistic Regression",
    X_train,
    Y_train,
    X_test,
    Y_test
)


/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/lib/python3.13/site-packages/sklearn/linear_model/_logisti


===== Logistic Regression =====
Training Time: 5.13 seconds
Inference Time: 0.03 seconds
Subset Accuracy: 0.2803
Micro F1: 0.4136
Macro F1: 0.2229

Classification Report:
                precision    recall  f1-score   support

    admiration       0.74      0.32      0.44       504
     amusement       0.84      0.45      0.58       264
         anger       0.72      0.11      0.19       198
     annoyance       0.75      0.02      0.04       320
      approval       0.73      0.06      0.12       351
        caring       0.73      0.08      0.15       135
     confusion       0.53      0.05      0.10       153
     curiosity       0.55      0.10      0.16       284
        desire       0.90      0.11      0.19        83
disappointment       1.00      0.01      0.01       151
   disapproval       0.53      0.04      0.07       267
       disgust       0.73      0.13      0.22       123
 embarrassment       0.00      0.00      0.00        37
    excitement       0.88      0.14      0.

Comparison of results

In [70]:
results_df = pd.DataFrame([
    {
        "Model": nb_results["model_name"],
        "Training Time": nb_results["train_time"],
        "Inference Time": nb_results["inference_time"],
        "Subset Accuracy": nb_results["subset_accuracy"],
        "Micro F1": nb_results["micro_f1"],
        "Macro F1": nb_results["macro_f1"]
    },
    {
        "Model": svm_results["model_name"],
        "Training Time": svm_results["train_time"],
        "Inference Time": svm_results["inference_time"],
        "Subset Accuracy": svm_results["subset_accuracy"],
        "Micro F1": svm_results["micro_f1"],
        "Macro F1": svm_results["macro_f1"]
    },
    {
        "Model": rf_results["model_name"],
        "Training Time": rf_results["train_time"],
        "Inference Time": rf_results["inference_time"],
        "Subset Accuracy": rf_results["subset_accuracy"],
        "Micro F1": rf_results["micro_f1"],
        "Macro F1": rf_results["macro_f1"]
    },
    {
        "Model": lr_results["model_name"],
        "Training Time": lr_results["train_time"],
        "Inference Time": lr_results["inference_time"],
        "Subset Accuracy": lr_results["subset_accuracy"],
        "Micro F1": lr_results["micro_f1"],
        "Macro F1": lr_results["macro_f1"]
    }
])

print(results_df)

                 Model  Training Time  Inference Time  Subset Accuracy  \
0          Naive Bayes       0.251737        0.029352         0.105399   
1           Linear SVM       5.115743        0.032705         0.351023   
2        Random Forest      19.855067        0.788837         0.030219   
3  Logistic Regression       5.127708        0.028880         0.280265   

   Micro F1  Macro F1  
0  0.179092  0.058784  
1  0.489610  0.345304  
2  0.068261  0.033785  
3  0.413614  0.222874  


Per-Emotion F1

In [71]:
from sklearn.metrics import f1_score

for i, emotion in enumerate(emotions):
    score = f1_score(
        Y_test.iloc[:, i],
        svm_results["Y_pred"][:, i]
    )
    print(emotion, score)

admiration 0.5667060212514758
amusement 0.6858407079646017
anger 0.3125
annoyance 0.17010309278350516
approval 0.22371364653243847
caring 0.24581005586592178
confusion 0.24630541871921183
curiosity 0.24289405684754523
desire 0.2807017543859649
disappointment 0.11976047904191617
disapproval 0.15294117647058825
disgust 0.38857142857142857
embarrassment 0.13043478260869565
excitement 0.34532374100719426
fear 0.5470085470085471
gratitude 0.9147982062780269
grief 0.0
joy 0.4380165289256198
love 0.7069767441860465
nervousness 0.0
optimism 0.49097472924187724
pride 0.21052631578947367
realization 0.13580246913580246
relief 0.16666666666666666
remorse 0.5473684210526316
sadness 0.46551724137931033
surprise 0.35353535353535354
neutral 0.5797184717035335


## Train Complex Models

Prepare Data

In [22]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score

In [28]:
model_name = "bert-base-uncased"
# Later use: "roberta-base"

train_df = goemotions_train[["clean_text"] + emotions].copy()
test_df = goemotions_test[["clean_text"] + emotions].copy()

train_df["labels"] = train_df[emotions].astype(float).values.tolist()
test_df["labels"] = test_df[emotions].astype(float).values.tolist()

train_dataset = Dataset.from_pandas(train_df[["clean_text", "labels"]])
test_dataset = Dataset.from_pandas(test_df[["clean_text", "labels"]])

In [29]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.remove_columns(["clean_text"])
test_dataset = test_dataset.remove_columns(["clean_text"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")

Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [30]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(emotions),
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [31]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "subset_accuracy": accuracy_score(labels, preds)
    }

In [32]:
training_args = TrainingArguments(
    output_dir="./bert_goemotions_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    logging_dir="./logs",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/Users/varunpanuganti/Desktop/NSF-REU-Project/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
bert_results = trainer.evaluate()
print(bert_results)